# Connect-4 PPO Evaluation Suite

This notebook evaluates your PPO agent with complementary lenses:

1. **Regret score (solver-ish):** compare your agent’s moves against a strong teacher (deep lookahead).
2. **Population rating:** compute a stable league rating (Elo).


In [1]:
MODEL_PATH = "PPO_Models/PPO_823.pt" 
#MODEL_PATH = "SupervisedModels/BASE_9.pt" 

In [2]:
# --- Imports & device ---
import os
import re
import math
import time
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Iterable, Any
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
import torch
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 666
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE, "| seed:", SEED)
start_time = time.time()

DEVICE: cuda | seed: 666


In [3]:
from C4.eval_oppo_dict import EVALUATION_OPPONENTS, EVAL_CFG
from C4.fast_connect4_lookahead import Connect4Lookahead
from C4.CNet192 import CNet192, load_cnet192
from PPO.actor_critic import ActorCritic, TransferCfg

LA_SHARED = Connect4Lookahead()
LA_SHARED.OPENING_RANDOM = False

print("OK: imported Connect4Lookahead, CNet192/load_cnet192, ActorCritic")


OK: imported Connect4Lookahead, CNet192/load_cnet192, ActorCritic


In [4]:
ROWS, COLS = 6, 7

def empty_board() -> np.ndarray:
    return np.zeros((ROWS, COLS), dtype=np.int8)

def legal_actions(board: np.ndarray) -> List[int]:
    return [c for c in range(COLS) if board[0, c] == 0]

def apply_action(board: np.ndarray, col: int, mark: int) -> np.ndarray:
    if board[0, col] != 0:
        raise ValueError(f"Column {col} is full")
    out = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if out[r, col] == 0:
            out[r, col] = np.int8(mark)
            return out
    raise RuntimeError("apply_action fell through")

def is_full(board: np.ndarray) -> bool:
    return bool(np.all(board[0, :] != 0))

def other_mark(mark: int) -> int:
    return 2 if mark == 1 else 1

def has_winner(board: np.ndarray, mark: int) -> bool:
    return bool(LA_SHARED.has_four(board, mark))

def terminal_outcome(board: np.ndarray) -> Optional[int]:
    if has_winner(board, 1): return 1
    if has_winner(board, 2): return 2
    if is_full(board): return 0
    return None

def board_to_pov_scalar(board012: np.ndarray, mark: int) -> np.ndarray:
    """(6,7) int8 in {-1,0,+1}, current player=+1."""
    b = np.asarray(board012, dtype=np.int8)
    me = np.int8(mark)
    opp = np.int8(other_mark(mark))
    pov = np.zeros_like(b, dtype=np.int8)
    pov[b == me] = 1
    pov[b == opp] = -1
    return pov


In [5]:
# --- Agent interface + baselines ---

class Agent:
    name: str = "Agent"
    def begin_episode(self, seed: int = 0) -> None:
        pass
    def select_action(self, board: np.ndarray, mark: int, rng: np.random.Generator) -> int:
        raise NotImplementedError

class RandomAgent(Agent):
    def __init__(self, name="Random"):
        self.name = name
    def select_action(self, board, mark, rng):
        legal = legal_actions(board)
        return int(rng.choice(legal)) if legal else 0

class LeftmostAgent(Agent):
    def __init__(self, name="Leftmost"):
        self.name = name
    def select_action(self, board, mark, rng):
        legal = legal_actions(board)
        return int(min(legal)) if legal else 0

class CenterAgent(Agent):
    def __init__(self, name="Center"):
        self.name = name
    def select_action(self, board, mark, rng):
        for c in [3,4,2,5,1,6,0]:
            if board[0, c] == 0:
                return int(c)
        return 0

class LookaheadAgent(Agent):
    def __init__(self, depth: int, name: Optional[str]=None, cfg: Optional[dict]=None):
        self.depth = int(depth)
        self.name = name or f"Lookahead-{depth}"
        self.la = Connect4Lookahead()
        self.la.OPENING_RANDOM = False
        if cfg:
            for k, v in cfg.items():
                setattr(self.la, k, v)
    def select_action(self, board, mark, rng):
        return int(self.la.n_step_lookahead(board, mark, depth=self.depth))

class ActorCriticAgent(Agent):
    """Wrap your PPO ActorCritic for this evaluation harness."""
    def __init__(self, ac: ActorCritic, name: str = "PPO-AC", temperature: float = 1.0, disable_rule_mixes: bool = True):
        self.ac = ac.to(DEVICE).eval()
        self.name = name
        self.temperature = float(temperature)
        self.disable_rule_mixes = bool(disable_rule_mixes)
        self._ply = 0

    def begin_episode(self, seed: int = 0) -> None:
        self._ply = 0
        torch.manual_seed(int(seed))
        if hasattr(self.ac, "begin_episode"):
            self.ac.begin_episode()
        if self.disable_rule_mixes and hasattr(self.ac, "set_phase_heuristics"):
            # Pure policy evaluation
            self.ac.set_phase_heuristics(center_start=0.0, guard_prob=0.0, win_now_prob=0.0)

    def select_action(self, board: np.ndarray, mark: int, rng: np.random.Generator) -> int:
        legal = legal_actions(board)
        if not legal:
            return 0
        pov = board_to_pov_scalar(board, mark)
        a, logp, v, info = self.ac.act(
            state_np=pov,
            legal_actions=legal,
            temperature=self.temperature,
            ply_idx=self._ply,
        )
        self._ply += 1
        return int(a)


In [6]:
# --- Load PPO checkpoint (save_cnet192 format) ---
def load_actor_critic_from_save_cnet192(path: str) -> ActorCritic:
    ac = ActorCritic.from_cnet192_checkpoint(
        path=path,
        device=DEVICE,
        override_cfg=None,
        transfer=TransferCfg(freeze_conv=False, strict_load=True),
    )
    return ac.to(DEVICE).eval()


ac = load_actor_critic_from_save_cnet192(MODEL_PATH)
AGENT_UNDER_TEST: Agent = ActorCriticAgent(ac, name=os.path.basename(MODEL_PATH), temperature=1.0, disable_rule_mixes=True)
print("Loaded PPO model:", MODEL_PATH)

Loaded PPO model: PPO_Models/PPO_823.pt


In [7]:
# --- Match runner (alternating starts) ---

@dataclass
class MatchStats:
    wins: int = 0
    losses: int = 0
    draws: int = 0
    @property
    def games(self) -> int:
        return self.wins + self.losses + self.draws
    @property
    def win_rate(self) -> float:
        return self.wins / self.games if self.games else 0.0
    @property
    def score(self) -> float:
        return (self.wins + 0.5 * self.draws) / self.games if self.games else 0.0

def play_game(agent1: Agent, agent2: Agent, seed: int, max_plies: int = 42) -> int:
    rng = np.random.default_rng(seed)
    torch.manual_seed(int(seed))
    agent1.begin_episode(seed=seed)
    agent2.begin_episode(seed=seed + 1)

    board = empty_board()
    mark = 1  # agent1 is mark=1, agent2 is mark=2
    for _ply in range(max_plies):
        if mark == 1:
            col = agent1.select_action(board, 1, rng)
            board = apply_action(board, col, 1)
            if has_winner(board, 1): return 1
        else:
            col = agent2.select_action(board, 2, rng)
            board = apply_action(board, col, 2)
            if has_winner(board, 2): return 2
        if is_full(board): return 0
        mark = other_mark(mark)
    return 0

def evaluate_matchup(agentA: Agent, agentB: Agent, n_games: int, seed: int = 0, alternate_starts: bool = True) -> MatchStats:
    st = MatchStats()
    for i in range(n_games):
        s = seed + i
        if alternate_starts and (i % 2 == 1):
            winner = play_game(agentB, agentA, seed=s)
            if winner == 1: st.losses += 1
            elif winner == 2: st.wins += 1
            else: st.draws += 1
        else:
            winner = play_game(agentA, agentB, seed=s)
            if winner == 1: st.wins += 1
            elif winner == 2: st.losses += 1
            else: st.draws += 1
    return st

st = evaluate_matchup(AGENT_UNDER_TEST, RandomAgent(), n_games=20, seed=SEED, alternate_starts=True)
print("Sanity vs Random:", st, "wr=", round(st.win_rate, 3), "score=", round(st.score, 3))


Sanity vs Random: MatchStats(wins=20, losses=0, draws=0) wr= 1.0 score= 1.0


In [8]:
%%time
def opponent_weight(label: str, base: float = 1.4, random_weight: float = 1.0, default_weight: float = 1.0) -> float:
    s = str(label)
    if "Random" in s:
        return float(random_weight)
    m = re.search(r"(\d+)", s)
    if m:
        return float(base) ** int(m.group(1))
    return float(default_weight)

def global_score_from_results(results: dict, base: float = 1.4) -> float:
    num = 0.0
    den = 0.0
    for label, stats in results.items():
        wr = float(stats.get("win_rate", 0.0))
        w = opponent_weight(label, base=base)
        num += w * wr
        den += w
    return num / den if den > 1e-12 else float("nan")

def build_opponent(label: str) -> Agent:
    if label == "Random": return RandomAgent()
    if label == "Leftmost": return LeftmostAgent()
    if label == "Center": return CenterAgent()
    if label.startswith("Lookahead-"):
        d = int(label.split("-")[-1])
        return LookaheadAgent(depth=d, name=label)
    raise ValueError(label)

def run_current_benchmark(agent: Agent, seed: int = 0) -> dict:
    results = {}
    items = list(EVALUATION_OPPONENTS.items())

    for label, n in tqdm(items, desc="Benchmark opponents"):
        opp = build_opponent(label)
        st = evaluate_matchup(agent, opp, n_games=int(n), seed=seed, alternate_starts=True)

        results[label] = {
            "wins": st.wins,
            "losses": st.losses,
            "draws": st.draws,
            "games": st.games,
            "win_rate": st.win_rate,
            "score": st.score,
        }

        tqdm.write(f"{agent.name:>18} vs {label:<12} | wr={st.win_rate:.3f} score={st.score:.3f} (n={st.games})")

    return results


results = run_current_benchmark(AGENT_UNDER_TEST, seed=SEED)
G = global_score_from_results(results, base=1.4)
print("\nGlobal score:", round(G, 3))


Benchmark opponents:   0%|          | 0/13 [00:00<?, ?it/s]

        PPO_823.pt vs Random       | wr=1.000 score=1.000 (n=200)
        PPO_823.pt vs Lookahead-1  | wr=0.860 score=0.860 (n=100)
        PPO_823.pt vs Lookahead-2  | wr=0.620 score=0.640 (n=100)
        PPO_823.pt vs Lookahead-3  | wr=0.490 score=0.490 (n=100)
        PPO_823.pt vs Lookahead-4  | wr=0.730 score=0.730 (n=100)
        PPO_823.pt vs Lookahead-5  | wr=0.500 score=0.510 (n=50)
        PPO_823.pt vs Lookahead-6  | wr=0.375 score=0.375 (n=24)
        PPO_823.pt vs Lookahead-7  | wr=0.800 score=0.800 (n=10)
        PPO_823.pt vs Lookahead-9  | wr=0.500 score=0.583 (n=6)
        PPO_823.pt vs Lookahead-11 | wr=0.750 score=0.750 (n=4)
        PPO_823.pt vs Lookahead-13 | wr=0.250 score=0.250 (n=4)
        PPO_823.pt vs Leftmost     | wr=0.980 score=0.980 (n=100)
        PPO_823.pt vs Center       | wr=1.000 score=1.000 (n=200)

Global score: 0.475
CPU times: total: 1min 33s
Wall time: 1min 39s


In [9]:
%%time
# --- Position dataset for regret scoring (with tqdm) ---

from tqdm.auto import tqdm
import random

def sample_positions_from_games(
    agent1: Agent,
    agent2: Agent,
    n_games: int,
    seed: int,
    min_ply: int = 4,
    max_ply: int = 30,
    per_game: int = 3,
    desc: str = "Sampling games",
) -> List[Tuple[np.ndarray, int]]:
    samples = []
    for g in tqdm(range(n_games), desc=desc, total=n_games):
        s = seed + g * 1000
        game_rng = np.random.default_rng(s)
        torch.manual_seed(int(s))

        agent1.begin_episode(seed=s)
        agent2.begin_episode(seed=s + 1)

        board = empty_board()
        mark = 1
        traj = []

        while True:
            traj.append((board.copy(), mark))
            if mark == 1:
                col = agent1.select_action(board, 1, game_rng)
                board = apply_action(board, col, 1)
                if has_winner(board, 1) or is_full(board):
                    break
            else:
                col = agent2.select_action(board, 2, game_rng)
                board = apply_action(board, col, 2)
                if has_winner(board, 2) or is_full(board):
                    break
            mark = other_mark(mark)

        lo = max(0, min_ply)
        hi = min(len(traj) - 1, max_ply)
        if hi <= lo:
            continue

        picks = game_rng.choice(
            np.arange(lo, hi + 1),
            size=min(per_game, hi - lo + 1),
            replace=False
        )
        for idx in picks:
            samples.append(traj[int(idx)])

    return samples

def find_trap_positions(
    n_target: int,
    seed: int,
    shallow: int = 1,
    deep: int = 9,
    min_ply: int = 6,
    max_ply: int = 28,
    score_gap: float = 800.0,
    max_search_games: int = 600,
) -> List[Tuple[np.ndarray, int]]:
    traps = []
    sh = Connect4Lookahead(); sh.OPENING_RANDOM = False
    dp = Connect4Lookahead(); dp.OPENING_RANDOM = False

    R = RandomAgent()
    checked = 0

    pbar = tqdm(range(max_search_games), desc="Searching trap positions", total=max_search_games)
    for g in pbar:
        samples = sample_positions_from_games(
            R, R,
            n_games=1,
            seed=seed + 9999 + g,
            min_ply=min_ply,
            max_ply=max_ply,
            per_game=5,
            desc="  trap game",  # nested tqdm (short)
        )

        for board, mark in samples:
            checked += 1
            legal = legal_actions(board)
            if not legal:
                continue

            sc_sh = sh.n_step_action_scores(board, mark, depth=shallow)
            sc_dp = dp.n_step_action_scores(board, mark, depth=deep)

            mv_sh = max(legal, key=lambda c: float(sc_sh[c]))
            mv_dp = max(legal, key=lambda c: float(sc_dp[c]))
            best_dp = max(float(sc_dp[c]) for c in legal)

            if mv_sh != mv_dp and (best_dp - float(sc_dp[mv_sh])) >= score_gap:
                traps.append((board, mark))
                pbar.set_postfix(found=len(traps), checked=checked)
                if len(traps) >= n_target:
                    tqdm.write(f"Found {len(traps)} traps after checking {checked} positions")
                    return traps

        pbar.set_postfix(found=len(traps), checked=checked)

    tqdm.write(f"Found {len(traps)} traps after checking {checked} positions")
    return traps

N_RANDOM_GAMES = 80
N_SELFPLAY_GAMES = 60
N_TRAPS = 120

pos_random = sample_positions_from_games(
    RandomAgent(), RandomAgent(),
    n_games=N_RANDOM_GAMES,
    seed=SEED + 1,
    desc="Sampling Random vs Random",
)

pos_self = sample_positions_from_games(
    LookaheadAgent(4), LookaheadAgent(4),
    n_games=N_SELFPLAY_GAMES,
    seed=SEED + 2,
    desc="Sampling LA4 vs LA4",
)

pos_traps = find_trap_positions(
    N_TRAPS,
    seed=SEED + 3,
    shallow=1,
    deep=9,
    score_gap=800.0,
    max_search_games=600,
)

dataset = pos_random + pos_self + pos_traps
random.shuffle(dataset)

print("Dataset sizes:", len(pos_random), len(pos_self), len(pos_traps), "=> total", len(dataset))


Sampling Random vs Random:   0%|          | 0/80 [00:00<?, ?it/s]

Sampling LA4 vs LA4:   0%|          | 0/60 [00:00<?, ?it/s]

Searching trap positions:   0%|          | 0/600 [00:00<?, ?it/s]

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

...

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

  trap game:   0%|          | 0/1 [00:00<?, ?it/s]

Found 120 traps after checking 1632 positions
Dataset sizes: 240 180 120 => total 540
CPU times: total: 47.5 s
Wall time: 49.6 s


In [10]:
# --- Regret / blunder metrics vs a strong teacher (deep lookahead) ---

TEACHER_DEPTH = 13
teacher = Connect4Lookahead()
teacher.OPENING_RANDOM = False

def best_moves_from_scores(scores: np.ndarray, legal: List[int], atol: float = 1e-6, rtol: float = 1e-6) -> Tuple[float, List[int]]:
    best = max(float(scores[c]) for c in legal)
    best_cols = [c for c in legal if np.isclose(float(scores[c]), best, atol=atol, rtol=rtol)]
    return best, best_cols

def regret_report(agent: Agent, dataset: List[Tuple[np.ndarray,int]], max_positions: int = 600, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    use = dataset[:max_positions] if len(dataset) > max_positions else dataset
    agent.begin_episode(seed=seed)

    for i, (board, mark) in enumerate(tqdm(use, desc=f"Regret labeling (teacher D={TEACHER_DEPTH})")):
        legal = legal_actions(board)
        if not legal: continue
        sc = teacher.n_step_action_scores(board, mark, depth=TEACHER_DEPTH)
        best, best_cols = best_moves_from_scores(sc, legal)

        a = agent.select_action(board, mark, rng)
        chosen = float(sc[a]) if a in legal else float("nan")
        regret = float(best - chosen) if np.isfinite(chosen) else float("inf")

        rows.append({
            "idx": i,
            "mark": int(mark),
            "agent_action": int(a),
            "best_cols": best_cols,
            "best_score": float(best),
            "chosen_score": chosen,
            "regret": regret,
            "is_optimal": bool(a in best_cols),
            "legal_count": int(len(legal)),
        })
    return pd.DataFrame(rows)

def summarize_regret(df: pd.DataFrame) -> dict:
    df2 = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["regret"])
    if len(df2) == 0:
        return {}
    med = float(np.median(df2["regret"].values))
    mad = float(np.median(np.abs(df2["regret"].values - med))) + 1e-12
    thr = med + 3.0 * mad
    return {
        "positions": int(len(df2)),
        "optimal_move_rate": float(df2["is_optimal"].mean()),
        "mean_regret": float(df2["regret"].mean()),
        "median_regret": med,
        "p90_regret": float(np.quantile(df2["regret"].values, 0.90)),
        "blunder_threshold": thr,
        "blunder_rate": float((df2["regret"] >= thr).mean()),
    }

df_reg = regret_report(AGENT_UNDER_TEST, dataset, max_positions=100, seed=SEED+10)
reg_summary = summarize_regret(df_reg)
reg_summary


Regret labeling (teacher D=13):   0%|          | 0/100 [00:00<?, ?it/s]

{'positions': 100,
 'optimal_move_rate': 0.53,
 'mean_regret': 17011041.288740236,
 'median_regret': 2.0,
 'p90_regret': 100000114.64734375,
 'blunder_threshold': 8.000000000003,
 'blunder_rate': 0.47}

In [11]:
%%time
# --- Cached "fake Elo" league ---
# Baseline opponents play each other ONCE and are cached.
# Each new AGENT_UNDER_TEST only plays vs baseline, then we recompute Elo from (baseline + new rows).
#
# Default games policy:
#   - LA >= 11 : 4 games
#   - LA >= 9  : 6 games
#   - else     : 10 games
#
# Files created:
#   league_cache/baseline_pairs.pkl
#   league_cache/vs__<agent_name_sanitized>.pkl

# from pathlib import Path
# import re
# import pandas as pd
# from tqdm.auto import tqdm

CACHE_DIR = Path("league_cache")
CACHE_DIR.mkdir(exist_ok=True)

BASELINE_CACHE = CACHE_DIR / "baseline_pairs.pkl"

def _sanitize(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(s))
    return s[:160]

def _la_depth_from_name(name: str) -> int:
    # Supports "Lookahead-7" and "LA-7"
    m = re.search(r"(?:Lookahead-|LA-)(\d+)", str(name))
    return int(m.group(1)) if m else 0

def games_for_pair(nameA: str, nameB: str) -> int:
    d = max(_la_depth_from_name(nameA), _la_depth_from_name(nameB))
    if d >= 11: return 4
    if d >= 9:  return 6
    return 10

def run_round_robin_cached(
    agents: list,
    seed: int,
    desc: str,
) -> pd.DataFrame:
    rows = []
    pairs = [(i, j) for i in range(len(agents)) for j in range(i + 1, len(agents))]
    for (i, j) in tqdm(pairs, desc=desc, total=len(pairs)):
        A, B = agents[i], agents[j]
        n_games = games_for_pair(A.name, B.name)
        st = evaluate_matchup(
            A, B,
            n_games=n_games,
            seed=seed + 5000 * (i * 100 + j),
            alternate_starts=True
        )
        rows.append((A.name, B.name, st.wins, st.losses, st.draws, n_games))
    return pd.DataFrame(rows, columns=["A", "B", "winsA", "lossesA", "draws", "games"])

def run_vs_baseline_cached(
    agent_under_test: Agent,
    baseline_agents: list,
    seed: int,
    desc: str,
) -> pd.DataFrame:
    # Cache per agent name (so rerunning notebook doesn't replay games)
    vs_path = CACHE_DIR / f"vs__{_sanitize(agent_under_test.name)}.pkl"
    if vs_path.exists():
        df = pd.read_pickle(vs_path)
        print(f"[cache] loaded vs-baseline: {vs_path} ({len(df)} rows)")
        return df

    rows = []
    for j, opp in enumerate(tqdm(baseline_agents, desc=desc, total=len(baseline_agents))):
        n_games = games_for_pair(agent_under_test.name, opp.name)
        st = evaluate_matchup(
            agent_under_test, opp,
            n_games=n_games,
            seed=seed + 10000 * j,
            alternate_starts=True
        )
        rows.append((agent_under_test.name, opp.name, st.wins, st.losses, st.draws, n_games))
    df = pd.DataFrame(rows, columns=["A", "B", "winsA", "lossesA", "draws", "games"])
    df.to_pickle(vs_path)
    print(f"[cache] saved vs-baseline: {vs_path} ({len(df)} rows)")
    return df

def elo_ratings_from_league(df_league: pd.DataFrame, k: float = 24.0, init: float = 1000.0) -> pd.DataFrame:
    agents = sorted(set(df_league["A"]).union(df_league["B"]))
    R = {a: float(init) for a in agents}

    def expected(ra, rb):
        return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    def update(a, b, score_a):
        ea = expected(R[a], R[b])
        R[a] += k * (score_a - ea)
        R[b] += k * ((1.0 - score_a) - (1.0 - ea))

    # Deterministic order: sort rows by (A,B) so Elo doesn't depend on append order
    df2 = df_league.sort_values(["A", "B"]).reset_index(drop=True)

    # Per-game updates (closer to classic Elo; still fast because games per row are small)
    for row in tqdm(df2.itertuples(index=False), desc="Elo updates", total=len(df2)):
        a, b, winsA, lossesA, draws, games = row
        winsA = int(winsA); lossesA = int(lossesA); draws = int(draws)
        for _ in range(winsA):   update(a, b, 1.0)
        for _ in range(lossesA): update(a, b, 0.0)
        for _ in range(draws):   update(a, b, 0.5)

    return (pd.DataFrame([{"agent": a, "elo": R[a]} for a in agents])
              .sort_values("elo", ascending=False)
              .reset_index(drop=True))

# ---------------- Build baseline league (cached) ----------------
baseline_agents = [
    RandomAgent(),
    LeftmostAgent(),
    CenterAgent(),
] + [LookaheadAgent(d) for d in range(1, 14)]

if BASELINE_CACHE.exists():
    df_base = pd.read_pickle(BASELINE_CACHE)
    print(f"[cache] loaded baseline: {BASELINE_CACHE} ({len(df_base)} rows)")
else:
    df_base = run_round_robin_cached(baseline_agents, seed=SEED + 30, desc="Baseline round-robin (cached)")
    df_base.to_pickle(BASELINE_CACHE)
    print(f"[cache] saved baseline: {BASELINE_CACHE} ({len(df_base)} rows)")

# ---------------- Evaluate new agent only vs baseline (cached per agent) ----------------
df_vs = run_vs_baseline_cached(AGENT_UNDER_TEST, baseline_agents, seed=SEED + 123, desc="Under-test vs baseline")

# ---------------- Combine + compute Elo ----------------
df_league = pd.concat([df_base, df_vs], ignore_index=True)
df_rating = elo_ratings_from_league(df_league, k=24.0, init=1000.0)

df_rating


[cache] loaded baseline: league_cache\baseline_pairs.pkl (120 rows)


Under-test vs baseline:   0%|          | 0/16 [00:00<?, ?it/s]

[cache] saved vs-baseline: league_cache\vs__PPO_823.pt.pkl (16 rows)


Elo updates:   0%|          | 0/136 [00:00<?, ?it/s]

CPU times: total: 1min 45s
Wall time: 1min 52s


,agent,elo
0,Lookahead-12,1299.272334
1,Lookahead-11,1293.130541
2,Lookahead-13,1258.549040
3,Lookahead-10,1215.849111
4,PPO_823.pt,1153.932953
5,Lookahead-8,1107.537129
6,Lookahead-9,1095.951151
7,Lookahead-5,1063.027796
8,Lookahead-6,1047.496290
9,Lookahead-7,1004.752746


In [12]:
df_league

,A,B,winsA,lossesA,draws,games
0,Random,Leftmost,4,6,0,10
1,Random,Center,2,8,0,10
2,Random,Lookahead-1,0,10,0,10
3,Random,Lookahead-2,0,10,0,10
4,Random,Lookahead-3,0,10,0,10
...,...,...,...,...,...,...
131,PPO_823.pt,Lookahead-9,4,2,0,6
132,PPO_823.pt,Lookahead-10,3,1,2,6
133,PPO_823.pt,Lookahead-11,0,3,1,4
134,PPO_823.pt,Lookahead-12,0,4,0,4


In [13]:
# --- Final combined report ---
print("Agent under test:", AGENT_UNDER_TEST.name)
print("\n1) Global score:", round(global_score_from_results(results, base=1.4), 5))

print("\n2) Regret summary (teacher depth=%d):" % TEACHER_DEPTH)
for k, v in reg_summary.items():
    print(f"   {k:>22}: {v}")

print("\n4) Population rating:")
print(df_rating.to_string(index=False))


Agent under test: PPO_823.pt

1) Global score: 0.475

2) Regret summary (teacher depth=13):
                positions: 100
        optimal_move_rate: 0.53
              mean_regret: 17011041.288740236
            median_regret: 2.0
               p90_regret: 100000114.64734375
        blunder_threshold: 8.000000000003
             blunder_rate: 0.47

4) Population rating:
       agent         elo
Lookahead-12 1299.272334
Lookahead-11 1293.130541
Lookahead-13 1258.549040
Lookahead-10 1215.849111
  PPO_823.pt 1153.932953
 Lookahead-8 1107.537129
 Lookahead-9 1095.951151
 Lookahead-5 1063.027796
 Lookahead-6 1047.496290
 Lookahead-7 1004.752746
 Lookahead-3  955.256295
 Lookahead-2  953.146164
 Lookahead-4  874.734949
 Lookahead-1  784.146552
      Center  769.863849
    Leftmost  647.785033
      Random  475.568068


In [14]:
# --- Export one-row evaluation summary to Excel ("Eval summary.xlsx") ---
#
# Keep only the useful stuff:
#   - headline: elo_like, global_score
#   - win-rates vs benchmark opponents: Random/Center/Leftmost + LA-*
#   - regret: reg_optimal_rate, reg_blunder_rate


from pathlib import Path
import pandas as pd
import numpy as np
import re

EXCEL_PATH = Path("Eval summary.xlsx")
SHEET_NAME = "Summary"

def _safe_get(d, key, default=np.nan):
    try:
        return d.get(key, default)
    except Exception:
        return default

def _label_to_col(label: str) -> str:
    """Map benchmark labels to friendly column names."""
    s = str(label)
    if s == "Random":
        return "Random"
    if s == "Center":
        return "Center"
    if s == "Leftmost":
        return "Leftmost"
    m = re.match(r"Lookahead-(\d+)$", s)
    if m:
        return f"LA-{int(m.group(1))}"
    return s.replace("wr_", "").replace("_", "-")

def _flatten_current_benchmark(results: dict) -> dict:
    """Flat columns with friendly names: Random, Center, Leftmost, LA-1 ..."""
    flat = {}
    for label, st in results.items():
        col = _label_to_col(label)
        flat[col] = float(_safe_get(st, "win_rate", np.nan))
    return flat

def _flatten_regret_minimal(reg_summary: dict) -> dict:
    return {
        "reg_optimal_rate": float(_safe_get(reg_summary, "optimal_move_rate", np.nan)),
        "reg_blunder_rate": float(_safe_get(reg_summary, "blunder_rate", np.nan)),
    }

def _get_elo_for_agent(df_rating: pd.DataFrame, agent_name: str) -> float:
    if df_rating is None or len(df_rating) == 0:
        return float("nan")
    if "agent" not in df_rating.columns:
        return float("nan")
    row = df_rating[df_rating["agent"] == agent_name]
    if len(row) == 0:
        return float("nan")
    if "conservative" in df_rating.columns:
        return float(row.iloc[0]["conservative"])
    if "elo" in df_rating.columns:
        return float(row.iloc[0]["elo"])
    return float("nan")

def _la_key(col: str) -> int:
    m = re.match(r"LA-(\d+)$", str(col))
    return int(m.group(1)) if m else 10**9

# ------------- Build one summary row -------------
agent_name = getattr(AGENT_UNDER_TEST, "name", "UnknownAgent")

row = {
    "elo_like": _get_elo_for_agent(df_rating if "df_rating" in globals() else None, agent_name),
    "global_score": float(G) if "G" in globals() else float("nan"),
}

if "results" in globals():
    row.update(_flatten_current_benchmark(results))

if "reg_summary" in globals():
    row.update(_flatten_regret_minimal(reg_summary))

df_new = pd.DataFrame([row], index=pd.Index([agent_name], name="agent"))

# ------------- Read existing -------------
if EXCEL_PATH.exists():
    try:
        df_old = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, index_col=0)
        df_old.index.name = "agent"
    except Exception:
        df_old = pd.DataFrame()
else:
    df_old = pd.DataFrame()

# Ensure consistent columns (union)
all_cols = sorted(set(df_old.columns).union(df_new.columns))
df_old = df_old.reindex(columns=all_cols)
df_new = df_new.reindex(columns=all_cols)

# Append time-series (duplicate index values allowed)
df_out = pd.concat([df_old, df_new], axis=0)

# ------------- Column order: elo, global, Random/Center/Leftmost, LA-1..LA-13, then regret -------------
cols = list(df_out.columns)

head = [c for c in ["elo_like", "global_score"] if c in cols]
basics = [c for c in ["Random", "Center", "Leftmost"] if c in cols]
la_cols = sorted([c for c in cols if re.match(r"LA-\d+$", str(c))], key=_la_key)

reg_cols = [c for c in ["reg_optimal_rate", "reg_blunder_rate"] if c in cols]
rest = [c for c in cols if c not in (head + basics + la_cols + reg_cols)]

ordered = head + basics + la_cols + reg_cols + rest
df_out = df_out.reindex(columns=ordered)

# ------------- Write Excel -------------
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as w:
    df_out.to_excel(w, sheet_name=SHEET_NAME, index=True)

print(f"Wrote {EXCEL_PATH} | rows now: {len(df_out)} | appended agent index: {agent_name}")


Wrote Eval summary.xlsx | rows now: 141 | appended agent index: PPO_823.pt


In [15]:
# --- Excel formatting: blue header, wider columns, filters, freeze panes (row + first column),
#     and number formatting (2 decimals) ---

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import datetime as _dt

EXCEL_PATH = "Eval summary.xlsx"
SHEET_NAME = "Summary"

wb = load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

# Header styling
header_fill = PatternFill(fill_type="solid", fgColor="1F4E79")  # dark-ish blue
header_font = Font(bold=True, color="FFFFFF")
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Apply style to header row
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_align

ws.row_dimensions[1].height = 22

# Freeze header row AND first column (agent index column)
ws.freeze_panes = "B2"

# Enable filters (auto-filter over the used range)
ws.auto_filter.ref = ws.dimensions

# Number formatting
# - Default numeric: 2 decimals
NUM_FMT_2DP = "0.000"

# Apply numeric format to all numeric cells (excluding header)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=2, max_col=ws.max_column):
    for cell in row:
        v = cell.value
        # Skip blanks and non-numbers; treat bool separately (bool is subclass of int in Python)
        if v is None or isinstance(v, bool):
            continue
        if isinstance(v, (int, float, np.integer, np.floating)):
            cell.number_format = NUM_FMT_2DP

# Column width auto-fit (approx; Excel has no true autofit via openpyxl)
min_w = 10
max_w = 45
pad = 2

for col_idx in range(1, ws.max_column + 1):
    col_letter = get_column_letter(col_idx)
    max_len = 0
    for row_idx in range(1, ws.max_row + 1):
        v = ws.cell(row=row_idx, column=col_idx).value
        if v is None:
            continue
        s = str(v)
        if len(s) > max_len:
            max_len = len(s)
    width = max(min_w, min(max_w, max_len + pad))
    ws.column_dimensions[col_letter].width = width

ws.sheet_view.showGridLines = True

wb.save(EXCEL_PATH)
print(f"Formatted: {EXCEL_PATH} ({SHEET_NAME})")


Formatted: Eval summary.xlsx (Summary)


In [16]:
end_time = time.time()
total_elapsed = (end_time - start_time) / 60
print(f"Evaluation completed in {total_elapsed:.1f} minutes")

Evaluation completed in 6.4 minutes
